<a href="https://colab.research.google.com/github/muhammeddanisht/langgraph-agents/blob/main/Langgraph_v7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V7 — Dual Memory + MCP Agent (Production-Grade)

**What this is:** A LangGraph agent combining short-term memory (per-conversation)
with long-term memory (per-user, persists across conversations) plus real MCP tool
integration (calculator, live weather).

**Architecture:** Custom 4-node StateGraph — `load_memories → agent → tools → save_memories`,
with conditional routing based on whether the LLM requests a tool call.

**Stack:** LangGraph · FastMCP 3.4.2 · langchain-mcp-adapters · Groq (llama-3.3-70b-versatile) · Python

**Builds on:** V5 (dual memory architecture) + V6 (MCP tool integration), merged into one agent.


In [ ]:
# Cell 1: Install all packages
# Same packages as V6 — no new ones needed for V7!
!pip install -q fastmcp langchain-mcp-adapters langchain-groq langgraph nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 749.2/749.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.2/221.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.0/170.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.9/272.9 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 17.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires 

In [ ]:
# Cell 2: All imports for V7
import os                    # to access environment variables (API key)
import threading             # to run MCP server in background (same as V6)
import time                  # to wait for server to start
import requests              # to call wttr.in for REAL weather data
import asyncio               # to handle async code in Colab
import json                  # to parse LLM's extracted facts as JSON
import uuid                  # to create unique keys when saving facts to store
import ast
import operator

import nest_asyncio          # fixes "asyncio cannot run nested" bug in Colab
nest_asyncio.apply()         # MUST run this before any asyncio.run() calls

from fastmcp import FastMCP  # to build our MCP server with tools

from langchain_groq import ChatGroq                          # Groq LLM (free)
from langchain_mcp_adapters.client import MultiServerMCPClient  # connects to our MCP server
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage  # message types

from langgraph.graph import StateGraph, START, END           # graph building blocks
from langgraph.graph.message import add_messages             # smart message accumulator
from langgraph.prebuilt import ToolNode                      # NEW IN V7: auto tool executor
from langgraph.checkpoint.memory import MemorySaver          # short-term memory
from langgraph.store.memory import InMemoryStore             # long-term memory
from langgraph.store.base import BaseStore                   # type hint for store
from langgraph.checkpoint.memory import InMemorySaver

from typing import Annotated, TypedDict                     # for defining State type

In [ ]:
# Cell 3: Load API key from Colab secrets (NEVER hardcode!)
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("API key loaded ✅")

API key loaded ✅


In [ ]:
# Cell 4a: Create FastMCP server
# This is the "tool registry" — we register tools to it below
mcp = FastMCP("v7_tools")

# v7_tools = name of this server (can be anything)
# mcp = the server object we'll add tools to

In [ ]:
# Cell 4b: Calculator MCP tool — AST-based safe evaluator (NO eval())

# Whitelist of math operators we allow — nothing else gets executed
SAFE_OPERATORS = {
    ast.Add: operator.add,        # +
    ast.Sub: operator.sub,        # -
    ast.Mult: operator.mul,       # *
    ast.Div: operator.truediv,    # /
    ast.Pow: operator.pow,        # **
    ast.Mod: operator.mod,        # %
    ast.FloorDiv: operator.floordiv,  # //
    ast.USub: operator.neg,       # unary minus, e.g. -5
    ast.UAdd: operator.pos,       # unary plus, e.g. +5
}

def safe_eval(expression: str):
    """Parse expression into a tree, then evaluate ONLY whitelisted operations."""
    tree = ast.parse(expression, mode='eval')
    # mode='eval' means: treat this as a single expression, not a full program
    # tree.body = the actual expression node (e.g. "347 * 28" becomes a BinOp node)
    return _eval_node(tree.body)

def _eval_node(node):
    """Recursively walk the tree. Reject anything not in our whitelist."""

    if isinstance(node, ast.Constant):
        # ast.Constant = a literal number like 347 or 28.5
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numbers allowed")

    elif isinstance(node, ast.BinOp):
        # BinOp = "binary operation" = something like "a OP b" (e.g. 347 * 28)
        op_type = type(node.op)
        if op_type not in SAFE_OPERATORS:
            raise ValueError(f"Operator not allowed: {op_type.__name__}")
        left = _eval_node(node.left)    # recursively evaluate left side
        right = _eval_node(node.right)  # recursively evaluate right side
        return SAFE_OPERATORS[op_type](left, right)

    elif isinstance(node, ast.UnaryOp):
        # UnaryOp = single-operand operation, e.g. "-5"
        op_type = type(node.op)
        if op_type not in SAFE_OPERATORS:
            raise ValueError(f"Operator not allowed: {op_type.__name__}")
        operand = _eval_node(node.operand)
        return SAFE_OPERATORS[op_type](operand)

    else:
        # Anything else (function calls, names, imports, attribute access)
        # gets REJECTED here. This is what blocks code injection.
        raise ValueError(f"Unsupported expression type: {type(node).__name__}")


@mcp.tool()
def calculator(expression: str) -> str:
    """Calculate a math expression. Example: '347 * 28' or '100 / 4 + 50'"""
    try:
        result = safe_eval(expression)   # FIXED: safe_eval, not eval()
        return f"Result: {result}"
    except Exception as e:
        return f"Calculator error: {str(e)}"

 Weather MCP tool — pulls REAL data from wttr.in (no API key needed)

In [ ]:
 #Weather MCP tool — pulls REAL data from wttr.in (no API key needed)
@mcp.tool()
def get_weather(city: str) -> str:
    """Get current weather for any city. Returns real live data."""
    try:
        url = f"https://wttr.in/{city}?format=3"  # format=3 = compact: "City: ☀️ +21°C"
        response = requests.get(url, timeout=5)    # 5 second timeout (safety)

        if response.status_code == 200:            # 200 = request succeeded
            return response.text.strip()           # .strip() removes extra spaces/newlines
        else:
            return f"Weather service returned error: {response.status_code}"
    except Exception as e:
        return f"Could not get weather: {str(e)}"  # catches network errors

 Start FastMCP server in background thread

In [ ]:
# Must run BEFORE building agent (agent needs server to be ready)
def run_server():
    """Function that starts our MCP server — runs in separate thread"""
    mcp.run(transport="streamable-http")   # streamable-http = Colab-compatible (NOT stdio)
                                           # Server starts at http://localhost:8000/mcp

server_thread = threading.Thread(
    target=run_server,   # what to run in this thread
    daemon=True          # daemon=True means: if main program stops, thread stops too
)
server_thread.start()   # launches the thread (server starts running in background)
time.sleep(2)           # wait 2 seconds for server to fully start before using it
print("✅ MCP server running at http://localhost:8000/mcp")

╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 3.4.2                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                  🖥  Server:      v7_tools, 3.4.2                             │                  
                 │                  🚀 Deploy free: https://horizon.prefect.io                  │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

[06/19/26 10:25:37] INFO     Starting MCP server 'v7_tools' with transport 'streamable-http' on    ]8;id=178186;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=682623;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py#304\304]8;;\
                             http://127.0.0.1:8000/mcp                                                             

INFO:     Started server process [10128]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


✅ MCP server running at http://localhost:8000/mcp


 Create our Groq LLM

In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # best free model on Groq (verified June 2026)
    temperature=0                      # temperature=0 = deterministic, consistent answers
)
print("✅ LLM ready")

✅ LLM ready


 Create BOTH memory layers (this is the V5 memory system!)

In [ ]:
#  Create BOTH memory layers
checkpointer = InMemorySaver()   # FIXED: canonical class name
store = InMemoryStore()

print("✅ Short-term memory (InMemorySaver) ready")
print("✅ Long-term memory (InMemoryStore) ready")

✅ Short-term memory (InMemorySaver) ready
✅ Long-term memory (InMemoryStore) ready


Agent State — what information flows through our graph

In [ ]:
# V7State has 3 fields (V5 had 2: messages + user_id)

class AgentState(TypedDict):
    # TypedDict = Python type that defines a dictionary's structure

    messages: Annotated[list, add_messages]
    # messages = list of all chat messages (HumanMessage, AIMessage, ToolMessage)
    # Annotated[list, add_messages] means:
    #   - it's a list
    #   - add_messages is the "reducer" — tells LangGraph HOW to update this field
    #   - add_messages ADDS new messages to existing list (doesn't replace)
    #   - without this, each update would ERASE old messages!

    user_id: str
    # user_id = which user is talking
    # used as key for long-term memory namespace
    # example: "dani" → all of Dani's facts stored under ("memories", "dani")

    memory_context: str
    # memory_context = facts loaded from InMemoryStore for this user
    # example: "- user is from Kerala\n- user likes Python"
    # loaded by load_memories node → read by agent node
    # reloaded fresh every turn (latest facts always available)

 load_memories — reads long-term facts from InMemoryStore

In [ ]:
# This node runs FIRST every turn, before LLM sees anything

def load_memories(state: AgentState) -> dict:
    """
    Read user's facts from long-term store.
    Put them in memory_context field of state.
    """

    # Get user_id from state (who is talking?)
    user_id = state.get("user_id", "default_user")
    # .get("user_id", "default_user") means:
    #   - try to get "user_id" from state dict
    #   - if not found, use "default_user" as fallback

    # Define namespace for this user's memories
    namespace = ("memories", user_id)
    # namespace = tuple used as "folder path" in the store
    # ("memories", "dani") = folder "memories" → subfolder "dani"
    # This separates Dani's facts from other users' facts

    # Search the store for ALL facts in this namespace
    memories = store.search(namespace)
    # store.search(namespace) = get ALL items stored under this namespace
    # Returns: list of Item objects, each has .value dict

    # Format facts into readable string
    if memories:
        # memories found! build string list
        facts = "\n".join([
            f"- {m.value.get('fact', '')}"  # each fact as "- user likes Python"
            for m in memories               # loop through each memory item
            if m.value.get('fact', '')      # skip empty facts (safety check)
        ])
        memory_context = facts
    else:
        # no memories yet for this user
        memory_context = "No facts stored yet for this user."

    # Return updated memory_context to state
    return {"memory_context": memory_context}
    # LangGraph takes this dict and updates state["memory_context"]

should_continue — decides what happens after agent responds

In [ ]:
# This is our custom router (replaces tools_condition for V7)
# V7 needs custom router because: after no tool call, we go to save_memories
# (not END directly like tools_condition would do)

def should_continue(state: AgentState) -> str:
    """
    Look at last AI message.
    If LLM called a tool → go to "tools" node.
    If LLM gave final answer → go to "save_memories" node.
    """

    messages = state["messages"]
    last_message = messages[-1]   # [-1] = last item in list = most recent message

    if last_message.tool_calls:
        # tool_calls = list of tools the LLM wants to call
        # if not empty = LLM is requesting tool use
        return "tools"            # go to ToolNode

    return "save_memories"        # LLM gave final answer → save any learned facts

# should_continue returns a STRING
# That string = name of the next node to go to
# LangGraph reads this string and routes accordingly

 save_memories — extract facts from conversation and save to store

In [ ]:
# This node runs LAST every turn, after LLM gives final answer
# It reads recent messages, asks LLM "what did we learn?", saves to store

def save_memories(state: AgentState) -> dict:
    """
    Use LLM to extract any new facts from this conversation.
    Save them to InMemoryStore for future conversations.
    """

    user_id = state.get("user_id", "default_user")  # who are we saving facts for?
    namespace = ("memories", user_id)                # their storage "folder"

    # Get last 6 messages (last 3 human-AI exchanges)
    # We don't need the WHOLE history — recent messages have the new info
    recent_messages = state["messages"][-6:]

    # Format messages into text for LLM to analyze
    conversation_text = ""
    for msg in recent_messages:
        if hasattr(msg, 'content') and msg.content:  # make sure message has content
            role = msg.__class__.__name__             # "HumanMessage", "AIMessage" etc.
            conversation_text += f"{role}: {msg.content}\n"

    # Ask LLM to extract facts from this conversation
    extraction_prompt = f"""Extract important personal facts about the user from this conversation.
Only extract NEW information (name, location, preferences, job, skills, goals, etc.)
Return ONLY a JSON array like: [{{"fact": "user is from Kerala"}}, {{"fact": "user likes Python"}}]
If no important facts found, return empty array: []
Do NOT include facts about tools used or calculations done.

Conversation:
{conversation_text}"""

    # Use LLM to extract facts (sync invoke, no tools needed here)
    response = llm.invoke([HumanMessage(content=extraction_prompt)])

    # Parse LLM response as JSON
    try:
        # response.content = the text the LLM returned
        # json.loads() = converts JSON text to Python list
        raw = response.content.strip()

        # Sometimes LLM wraps JSON in ```json ... ``` code blocks
        # Clean that out before parsing
        if "```json" in raw:
            raw = raw.split("```json")[1].split("```")[0].strip()
        elif "```" in raw:
            raw = raw.split("```")[1].split("```")[0].strip()

        extracted_facts = json.loads(raw)   # parse: '[{"fact": "..."}]' → Python list

        # Save each fact to the store
        for fact_obj in extracted_facts:
            fact_text = fact_obj.get("fact", "")   # extract "fact" value from dict
            if fact_text:                           # only save non-empty facts
                # Create unique key for this fact
                fact_key = f"fact_{uuid.uuid4().hex[:8]}"
                # uuid4() = random UUID like "a3f9c2d1f7b8e1d2"
                # .hex[:8] = take first 8 characters = "a3f9c2d1"
                # f"fact_{...}" = "fact_a3f9c2d1" ← unique key

                # Save to store
                store.put(namespace, fact_key, {"fact": fact_text})
                # store.put(namespace, key, value_dict)
                # namespace = ("memories", "dani")
                # key = "fact_a3f9c2d1"
                # value = {"fact": "user is from Kerala"}
                print(f"💾 Saved fact: {fact_text}")

    except (json.JSONDecodeError, Exception) as e:
        # If LLM didn't return valid JSON, just skip saving
        # Better to save nothing than to crash the whole agent
        pass

    return {}  # no state changes from this node — it's a write-only node

 Async function that builds the complete V7 agent

In [ ]:
# Must be async because: getting MCP tools requires async (await client.get_tools())
# Everything that needs the tools must be INSIDE this function

async def build_agent():
    """
    1. Connect to MCP server
    2. Get tools
    3. Bind tools to LLM
    4. Define agent node (needs bound LLM)
    5. Build StateGraph
    6. Compile with MemorySaver + InMemoryStore
    7. Return compiled agent
    """

    # --- STEP 1: Connect to MCP server and get tools ---
    client = MultiServerMCPClient({
        "my_tools": {
            "transport": "http",               # same as V6!
            "url": "http://localhost:8000/mcp" # where our FastMCP server is running
        }
    })

    tools = await client.get_tools()  # await = wait for async operation
    # get_tools() = fetches tool schemas from MCP server
    # Returns: list of LangChain BaseTool objects (calculator, get_weather)
    # "await" is needed because network call takes time

    print(f"✅ Got {len(tools)} MCP tools: {[t.name for t in tools]}")

    # --- STEP 2: Bind tools to LLM ---
    bound_llm = llm.bind_tools(tools)
    # bind_tools() = tells LLM "these tools exist, you can call them"
    # bound_llm = same LLM but now knows about calculator + get_weather
    # Without this: LLM can't call any tools

    # --- STEP 3: Define agent node (needs bound_llm, defined INSIDE this function) ---
    def call_agent(state: AgentState) -> dict:
        """
        The main LLM node.
        Reads: memory_context + chat messages
        Calls: bound_llm (LLM that knows about tools)
        Returns: LLM's response (might have tool_calls or final answer)
        """

        # Build system message with user's facts
        memory_context = state.get("memory_context", "")
        user_id = state.get("user_id", "default_user")

        if memory_context and memory_context != "No facts stored yet for this user.":
            system_content = f"""You are a helpful AI assistant with memory and tools.

Known facts about {user_id}:
{memory_context}

You have access to: calculator (for math) and get_weather (for real weather data).
Use tools when the user asks for calculations or weather information."""
        else:
            system_content = f"""You are a helpful AI assistant with memory and tools.
You have access to: calculator (for math) and get_weather (for real weather data).
Use tools when needed."""

        # Get only chat messages (not system messages) from state
        chat_messages = [
            msg for msg in state["messages"]
            if not isinstance(msg, SystemMessage)  # filter out old system messages
        ]

        # Full message list for LLM: [system_msg] + [all chat history]
        full_messages = [SystemMessage(content=system_content)] + chat_messages

        # Call LLM
        response = bound_llm.invoke(full_messages)
        # bound_llm.invoke() = sends messages to Groq, gets response
        # response = AIMessage with either:
        #   a) tool_calls → LLM wants to use a tool
        #   b) content → LLM has final answer

        return {"messages": [response]}
        # add_messages reducer APPENDS this to state["messages"]

    # --- STEP 4: Build StateGraph ---
    builder = StateGraph(AgentState)
    # StateGraph(AgentState) = create new graph that uses AgentState as its state type

    # Add all nodes to the graph
    builder.add_node("load_memories", load_memories)   # node 1: reads store
    builder.add_node("agent", call_agent)              # node 2: LLM call
    builder.add_node("tools", ToolNode(tools))         # node 3: executes tool calls
    builder.add_node("save_memories", save_memories)   # node 4: writes to store

    # ToolNode(tools) explanation:
    # ToolNode = prebuilt LangGraph node
    # Pass it your tools list: ToolNode(tools)
    # It automatically: reads tool_calls from AI message → runs the tool → returns ToolMessage
    # No custom code needed!

    # Add edges (the arrows connecting nodes)
    builder.add_edge(START, "load_memories")          # always start by loading memories
    builder.add_edge("load_memories", "agent")        # then call LLM

    builder.add_conditional_edges(
        "agent",          # from agent node
        should_continue,  # call this function to decide where to go
        {
            "tools": "tools",              # if returns "tools" → go to tools node
            "save_memories": "save_memories"  # if returns "save_memories" → go there
        }
    )
    # add_conditional_edges = "dynamic routing"
    # Not a fixed path — depends on what should_continue() returns

    builder.add_edge("tools", "agent")        # after tools run → back to agent
    builder.add_edge("save_memories", END)    # after saving → done

    # --- STEP 5: Compile graph with BOTH memory layers ---
    agent = builder.compile(
        checkpointer=checkpointer,   # short-term: MemorySaver (thread_id scoped)
        store=store                  # long-term: InMemoryStore (user_id scoped)
    )
    # compile() = build the actual runnable graph
    # checkpointer= → graph saves state after every node (short-term memory)
    # store= → graph knows about long-term store (accessible inside nodes)

    print("✅ V7 Agent compiled successfully!")
    print("Memory + MCP Tools = Complete Production Agent")

    return agent

 Actually run the build function (async → use asyncio.run)

In [ ]:
# nest_asyncio.apply() (Cell 2) makes asyncio.run() work inside Colab

agent = asyncio.run(build_agent())
# asyncio.run() = run an async function from sync code
# After this line: agent is ready to use
print("🤖 Agent is ready!")

INFO:     127.0.0.1:59228 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59246 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59234 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:59258 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59270 - "DELETE /mcp HTTP/1.1" 200 OK
✅ Got 2 MCP tools: ['calculator', 'get_weather']
✅ V7 Agent compiled successfully!
Memory + MCP Tools = Complete Production Agent
🤖 Agent is ready!


 ask_agent helper function

In [ ]:
# Wraps the graph invocation in a clean, easy-to-use function

async def ask_agent(question: str, user_id: str = "dani", thread_id: str = "thread_1"):
    """
    Send a question to the V7 agent.

    question = what the user is asking
    user_id  = who is asking (for long-term memory)
    thread_id = which conversation session (for short-term memory)
    """

    config = {
        "configurable": {
            "thread_id": thread_id   # tells MemorySaver which thread to use
            # user_id goes in STATE (not config) for V7
        }
    }

    # ainvoke = async version of invoke (needed for MCP async tools)
    result = await agent.ainvoke(
        input={
            "messages": [HumanMessage(content=question)],  # user's question
            "user_id": user_id,           # who is talking (for long-term memory)
            "memory_context": ""          # will be filled by load_memories node
        },
        config=config                     # thread_id for short-term memory
    )

    # Get the last message = agent's final response
    return result["messages"][-1].content

 Test 1 — Calculator tool

In [ ]:
# Expectation: agent uses MCP calculator tool, gets REAL math result

answer = asyncio.run(ask_agent("What is 347 multiplied by 28?", user_id="dani"))
print(f"Agent: {answer}")

# Expected: agent calls calculator tool → eval("347 * 28") → 9716
# Should say "347 × 28 = 9716" or similar

INFO:     127.0.0.1:59286 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59302 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:59316 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59326 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59334 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59340 - "DELETE /mcp HTTP/1.1" 200 OK
Agent: The result of 347 multiplied by 28 is 9716.


Test 2 — Weather tool

In [ ]:
answer = asyncio.run(ask_agent("What is the weather in Kozhikode?", user_id="dani"))
print(f"Agent: {answer}")

# Expected: agent calls get_weather("Kozhikode") → wttr.in returns REAL weather

INFO:     127.0.0.1:59344 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59348 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:59364 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59376 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59390 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59394 - "DELETE /mcp HTTP/1.1" 200 OK
💾 Saved fact: user is from Kozhikode
Agent: The current weather in Kozhikode is 81°F with a chance of rain.


Test 3 — Long-term memory (THE KEY V7 FEATURE)

In [ ]:
# Step 1: Tell agent your name (in thread_1)
print("=== TURN 1: Tell agent your name ===")
answer1 = asyncio.run(ask_agent(
    "My name is Dani and I am learning AI engineering.",
    user_id="dani",
    thread_id="thread_1"
))
print(f"Agent: {answer1}")

# Step 2: Start NEW thread (thread_2) — short-term memory resets!
# BUT long-term memory (InMemoryStore) still has your name
print("\n=== TURN 2: New thread — does agent remember? ===")
answer2 = asyncio.run(ask_agent(
    "What is my name and what am I learning?",
    user_id="dani",
    thread_id="thread_2"   # DIFFERENT thread_id
))
print(f"Agent: {answer2}")

# Expected in TURN 2: Agent DOES remember Dani's name
# Because: save_memories saved it to InMemoryStore in thread_1
# And: load_memories loaded it at start of thread_2
# This is the proof that long-term memory works!

=== TURN 1: Tell agent your name ===
💾 Saved fact: user's name is Dani
💾 Saved fact: user is learning AI engineering
💾 Saved fact: user is from Kozhikode
Agent: Nice to meet you, Dani. It's great that you're learning AI engineering. That's a fascinating field with a lot of potential for growth and innovation. If you have any questions or need help with anything related to AI engineering, feel free to ask. I'm here to assist you. By the way, how's the weather in Kozhikode treating you today?

=== TURN 2: New thread — does agent remember? ===
💾 Saved fact: user's name is Dani
💾 Saved fact: user is learning AI engineering
Agent: Your name is Dani, and you are learning AI engineering.
